In [2]:
import pandas as pd
import pyomo.environ as pyo
import time
from typing import Dict, List, Set, Tuple

### 1.Configuration & File Paths

In [3]:
COLLEGES_FILE = 'ties_colleges.csv'
SUBJECT_GROUPS_FILE = 'ties_groups.csv'
APPLICATIONS_FILE = 'ties_applications.csv'

### 2.Data loading

In [4]:
def load_and_prepare_data() -> Tuple[
    pd.DataFrame,          # colleges
    pd.DataFrame,          # subject groups
    pd.DataFrame,          # applications
    Dict[str, int],        # quota_capacity: quota_id -> capacity
    Dict[int, List[str]],  # college_quotas: college_id -> list of quota_ids
    List[Tuple[int, int]], # applications list (student, college)
    Dict[Tuple[int, int], int],   # rank_dict: (student, college) -> rank
    Dict[Tuple[int, int], float], # score_dict: (student, college) -> score
    Set[int],              # student_set
    int,                   # K_VAL (max rank + 1)
    Dict[str, Set[int]],   # quota_applicants: quota_id -> set of student IDs
    Dict[str, List[float]] # S_p_dict: quota_id -> sorted unique scores
]:
    """
    Load CSV files and build all necessary data structures for the COM-Chilean model.
    Returns all core data objects.
    """
    df_colleges = pd.read_csv(COLLEGES_FILE)
    df_subject_groups = pd.read_csv(SUBJECT_GROUPS_FILE)
    df_apps = pd.read_csv(APPLICATIONS_FILE)

    # -------- Build quota capacities and college-to-quota mapping --------
    quota_capacity: Dict[str, int] = {}
    college_quotas: Dict[int, List[str]] = {}

    # Individual college quotas (C_c)
    for _, row in df_colleges.iterrows():
        cid = int(row['college_id'])
        q_id = f"C_{cid}"
        quota_capacity[q_id] = int(row['capacity'])
        college_quotas.setdefault(cid, []).append(q_id)

    # Subject group quotas (S_g)
    for _, row in df_subject_groups.iterrows():
        q_id = f"S_{int(row['group_id'])}"
        quota_capacity[q_id] = int(row['group_capacity'])

    # Link each college to its subject group quota
    for _, row in df_colleges.iterrows():
        cid = int(row['college_id'])
        q_id = f"S_{int(row['subject_id'])}"
        if q_id in quota_capacity:
            college_quotas[cid].append(q_id)

    # -------- Process applications --------
    applications: List[Tuple[int, int]] = []
    rank_dict: Dict[Tuple[int, int], int] = {}
    score_dict: Dict[Tuple[int, int], float] = {}
    student_set: Set[int] = set()
    student_choices: Dict[int, List[int]] = {}

    for _, row in df_apps.iterrows():
        i = int(row['student_id'])
        j = int(row['college_id'])
        rank = int(row['rank'])
        score = float(row['score'])

        student_set.add(i)
        student_choices.setdefault(i, []).append(j)
        applications.append((i, j))
        rank_dict[(i, j)] = rank
        score_dict[(i, j)] = score

    K_VAL = max(rank_dict.values()) + 1

    # -------- For each quota, collect applicants and their scores --------
    quota_applicants: Dict[str, Set[int]] = {q: set() for q in quota_capacity}
    quota_score_set: Dict[str, Set[float]] = {q: set() for q in quota_capacity}

    for (i, j) in applications:
        score = score_dict[(i, j)]
        for q in college_quotas[j]:
            quota_applicants[q].add(i)
            quota_score_set[q].add(score)

    # Sorted unique scores per quota
    S_p_dict: Dict[str, List[float]] = {
        q: sorted(list(scores)) for q, scores in quota_score_set.items() if scores
    }

    return (
        df_colleges,
        df_subject_groups,
        df_apps,
        quota_capacity,
        college_quotas,
        applications,
        rank_dict,
        score_dict,
        student_set,
        K_VAL,
        quota_applicants,
        S_p_dict
    )

### 3.Pyomo Model Construction

In [5]:
def build_com_model(
    student_set: Set[int],
    applications: List[Tuple[int, int]],
    quota_capacity: Dict[str, int],
    college_quotas: Dict[int, List[str]],
    rank_dict: Dict[Tuple[int, int], int],
    score_dict: Dict[Tuple[int, int], float],
    S_p_dict: Dict[str, List[float]],
    quota_applicants: Dict[str, Set[int]],
    K_VAL: int
) -> pyo.ConcreteModel:
    """
    Build and return the Pyomo ConcreteModel for the COM-Chilean mechanism.
    All constraints and the objective are implemented exactly as in the original code.
    """
    model = pyo.ConcreteModel(name="COM_Chilean")

    # -------- Sets --------
    model.students = pyo.Set(initialize=list(student_set))
    model.applications = pyo.Set(initialize=applications, dimen=2)  # (student, college)

    model.quotas = pyo.Set(initialize=list(quota_capacity.keys()))

    # T_set: (quota, score) pairs for which score exists
    T_indices = [(q, s) for q in S_p_dict for s in S_p_dict[q]]
    model.quota_scores = pyo.Set(initialize=T_indices, dimen=2)

    # D_bar_set: (student, quota) where student applied to at least one college in that quota
    D_bar_indices = [(i, q) for q in S_p_dict for i in quota_applicants[q]]
    model.student_quotas = pyo.Set(initialize=D_bar_indices, dimen=2)

    # -------- Parameters --------
    model.quota_capacity = pyo.Param(model.quotas, initialize=quota_capacity)
    model.rank = pyo.Param(model.applications, initialize=rank_dict)
    model.score = pyo.Param(model.applications, initialize=score_dict)

    # -------- Variables --------
    model.x = pyo.Var(model.applications, domain=pyo.Binary)          # x_ij
    model.t = pyo.Var(model.quota_scores, domain=pyo.Binary)          # t_p^s
    model.d_bar = pyo.Var(model.student_quotas, domain=pyo.Binary)    # \bar{d}_i^p

    # -------- Objective (maximize student-optimal stable matching) --------
    def obj_rule(m):
        return sum((K_VAL - m.rank[i, j]) * m.x[i, j] for (i, j) in m.applications)
    model.Objective = pyo.Objective(rule=obj_rule, sense=pyo.maximize)

    # -------- Constraint (1): each student assigned to at most one college --------
    def student_capacity_rule(m, i):
        choices = [j for (s, j) in m.applications if s == i]
        return sum(m.x[i, j] for j in choices) <= 1
    model.student_capacity = pyo.Constraint(model.students, rule=student_capacity_rule)

    # -------- Constraint (32): admissibility via quota cutoffs (Chilean policy: no (26)) --------
    model.admissibility = pyo.ConstraintList()
    for (i, j) in applications:
        for q in college_quotas[j]:
            s_val = score_dict[(i, j)]
            model.admissibility.add(model.x[i, j] <= model.t[q, s_val])

    # -------- Constraint (33): cutoff monotonicity per quota --------
    model.cutoff_monotonicity = pyo.ConstraintList()
    for q, scores in S_p_dict.items():
        for k in range(len(scores) - 1):
            model.cutoff_monotonicity.add(model.t[q, scores[k]] <= model.t[q, scores[k+1]])

    # -------- Constraint (34): network envy-freeness --------
    def envy_free_rule(m, i, j):
        # Better or equal ranked colleges from i's perspective
        better_or_equal = [k for k in [c for (s, c) in m.applications if s == i] 
                           if m.rank[i, k] <= m.rank[i, j]]
        sum_x_better = sum(m.x[i, k] for k in better_or_equal)
        sum_cutoffs = sum(1 - m.t[q, m.score[i, j]] for q in college_quotas[j])
        return 1 <= sum_x_better + sum_cutoffs
    model.envy_free = pyo.Constraint(model.applications, rule=envy_free_rule)

    # -------- Constraint (42): \bar{d}_i^p only if accepted to quota p --------
    def d_bar_acceptance_rule(m, i, q):
        relevant = [j for (s, j) in m.applications if s == i and q in college_quotas[j]]
        if not relevant:
            return m.d_bar[i, q] <= 0
        return m.d_bar[i, q] <= sum(m.x[i, j] for j in relevant)
    model.d_bar_acceptance = pyo.Constraint(model.student_quotas, rule=d_bar_acceptance_rule)

    # -------- Constraint (43): \bar{d}_i^p bounded by cutoff difference --------
    model.d_bar_cutoff = pyo.ConstraintList()
    for (i, q) in model.student_quotas:
        relevant_j = [j for (s, j) in applications if s == i and q in college_quotas[j]]
        if relevant_j:
            j0 = relevant_j[0]  # score is same for all colleges in the quota
            s_val = score_dict[(i, j0)]
            scores = S_p_dict[q]
            k_idx = scores.index(s_val)
            if k_idx == 0:
                model.d_bar_cutoff.add(model.d_bar[i, q] <= model.t[q, s_val])
            else:
                s_prev = scores[k_idx - 1]
                model.d_bar_cutoff.add(model.d_bar[i, q] <= model.t[q, s_val] - model.t[q, s_prev])

    # -------- Constraint (41): Chilean non-wastefulness (capacity can be exceeded by d_bar) --------
    def chilean_non_wasteful_rule(m, q):
        if q not in S_p_dict:
            return pyo.Constraint.Skip
        relevant_apps = [(i, j) for (i, j) in applications if q in college_quotas[j]]
        sum_x = sum(m.x[i, j] for (i, j) in relevant_apps)
        sum_d = sum(m.d_bar[i, q] for i in quota_applicants[q])
        return (sum_x - sum_d) <= m.quota_capacity[q] - 1
    model.chilean_non_wasteful = pyo.Constraint(model.quotas, rule=chilean_non_wasteful_rule)

    return model

### 4. Solve and Report

In [6]:
def solve_and_report(model: pyo.ConcreteModel, 
                     student_set: Set[int],
                     applications: List[Tuple[int, int]],
                     rank_dict: Dict[Tuple[int, int], int],
                     score_dict: Dict[Tuple[int, int], float],
                     quota_capacity: Dict[str, int],
                     S_p_dict: Dict[str, List[float]],
                     college_quotas: Dict[int, List[str]]) -> None:
    """
    Solve the model with GLPK, print results and quota violations.
    """
    print("\nSolving the model with GLPK...")
    start_time = time.time()
    solver = pyo.SolverFactory('glpk')
    results = solver.solve(model, tee=True)
    end_time = time.time()
    solve_time = end_time - start_time

    if results.solver.termination_condition != pyo.TerminationCondition.optimal:
        print(f"\nSolver failed. Status: {results.solver.termination_condition}")
        return

    print(f"\n*** Optimal Solution Found in {solve_time:.4f} seconds! ***")

    # Collect assignments
    assignments = []
    assignment_tuples = []
    for (i, j) in applications:
        if pyo.value(model.x[i, j]) > 0.5:
            assignments.append({
                'student_id': i,
                'college_id': j,
                'rank': rank_dict[(i, j)],
                'score': score_dict[(i, j)]
            })
            assignment_tuples.append((i, j))

    df_results = pd.DataFrame(assignments).sort_values('student_id').reset_index(drop=True)
    assignment_tuples.sort()

    print(f"Total Students Assigned: {len(df_results)} out of {len(student_set)}")
    print(f"Solver Time: {solve_time:.4f} seconds")
    print(f"Solution Signature (Hash): {hash(tuple(assignment_tuples))}")

    # --- Quota violation report ---
    print("\n--- Quota Violations (Chilean Policy in Action) ---")
    for q in S_p_dict.keys():
        relevant_apps = [(i, j) for (i, j) in applications if q in college_quotas[j]]
        admitted = sum(1 for i, j in relevant_apps if pyo.value(model.x[i, j]) > 0.5)
        q_type = "College" if q.startswith("C") else "Subject"
        q_name = q.split("_")[1]
        status = "VIOLATED!" if admitted > quota_capacity[q] else "OK"
        print(f"{q_type} {q_name:2} | Admitted: {admitted} | Limit: {quota_capacity[q]} -> {status}")

### 5. Execute

In [9]:
print("Loading Data for COM-Chilean Model (Full Dataset)...")
(df_colleges, df_subject_groups, df_apps, quota_capacity,
 college_quotas, applications, rank_dict, score_dict,
 student_set, K_VAL, quota_applicants, S_p_dict) = load_and_prepare_data()

print(f"Data Ready: {len(student_set)} Students, {len(df_colleges)} Colleges, "
      f"{len(df_subject_groups)} Common Quotas.")
print(f"Total Applications (E): {len(applications)}")

print("\nBuilding Pyomo Model... (COM-Chilean)")
model = build_com_model(
    student_set,
    applications,
    quota_capacity,
    college_quotas,
    rank_dict,
    score_dict,
    S_p_dict,
    quota_applicants,
    K_VAL
)

solve_and_report(
    model,
    student_set,
    applications,
    rank_dict,
    score_dict,
    quota_capacity,
    S_p_dict,
    college_quotas
)

Loading Data for COM-Chilean Model (Full Dataset)...
Data Ready: 15 Students, 5 Colleges, 4 Common Quotas.
Total Applications (E): 60

Building Pyomo Model... (COM-Chilean)

Solving the model with GLPK...
GLPSOL--GLPK LP/MIP Solver 5.0
Parameter(s) specified in the command line:
 --write /tmp/tmpyvu2h_kw.glpk.raw --wglp /tmp/tmpbg8nqf8g.glpk.glp --cpxlp
 /tmp/tmp_xv7qend.pyomo.lp
Reading problem data from '/tmp/tmp_xv7qend.pyomo.lp'...
/tmp/tmp_xv7qend.pyomo.lp:3393: warning: lower bound of variable 'x2' redefined
/tmp/tmp_xv7qend.pyomo.lp:3393: warning: upper bound of variable 'x2' redefined
511 rows, 264 columns, 1526 non-zeros
264 integer variables, all of which are binary
3657 lines were read
Writing problem data to '/tmp/tmpbg8nqf8g.glpk.glp'...
2875 lines were written
GLPK Integer Optimizer 5.0
511 rows, 264 columns, 1526 non-zeros
264 integer variables, all of which are binary
Preprocessing...
8 hidden covering inequaliti(es) were detected
502 rows, 264 columns, 1294 non-zeros
2